In [108]:
import requests
from bs4 import BeautifulSoup
import csv
import os
import logging
import time
import random
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import smtplib
from email.message import EmailMessage

In [109]:
%%writefile .env
KILIMALL_EMAIL_SENDER='hudsonprodigy26@gmail.com'
KILIMALL_EMAIL_PASSWORD='hudson@ca6'
KILIMALL_EMAIL_RECEIVER='hudsonprodigy40@gmail.com'

Overwriting .env


In [110]:
!pip install python-dotenv pandas

In [111]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('kilimall_scraper.log'),
        logging.StreamHandler()
    ]
)

load_dotenv()

EMAIL_PASSWORD = os.getenv('KILIMALL_EMAIL_PASSWORD')
EMAIL_SENDER = os.getenv('KILIMALL_EMAIL_SENDER')
EMAIL_RECEIVER = os.getenv('KILIMALL_EMAIL_RECEIVER')

In [112]:
def fetch_kilimall_page(url,max_retries=5):
    user_agents = [
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    ]    
    headers= {
       'User-Agent' : random.choice(user_agents),
       'Accept' : 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
       'Accept-Language': 'en-US,en;q=0.5',
       'Accept-Encoding': 'gzip, deflate, br',
       'connection': 'keep-alive',
       'Upgrade-Insecure-Requests':'1'
   }        
      
   
    for attempt in range(1, max_retries + 1):
        try:
            logging.info(f"Attempt {attempt} to fetch: {url}")
            response = requests.get(url, headers=headers, timeout=15)
            response.raise_for_status()
            response.encoding = 'utf-8'
            soup = BeautifulSoup(response.text, 'html.parser')
            logging.info(f"Successfully fetched page: {url}")
            return soup
        except requests.exceptions.RequestException as e:
            logging.warning(f"Attempt {attempt} failed:{e}")
            if attempt == max_retries:
                logging.error(f"All {max_retries} attempts failed for {url}")
                return None
            wait_time = 2 * attempt
            logging.info(f"Waiting {Wait_time} seconds before retry.....")
            time.sleep(wait_time)
    return None
            

In [113]:
def extract_kilimall_products(soup):
    products_list = []

    if soup is None:
        logging.warnimg("Soup is None,returning empty list")
        return produts_list

    items = soup.find_all('div',class_='listing-item')

    if not items:
        logging.warning("No 'listing-item' found.Trying 'product-item'....")
        items = soup.find_all('div',class_='product-item')

        
    if not items:
        logging.warning("No product containers found at all.Page might be blocked or chaanged.")
        return products_list
        
    logging.info(f"Found {len(items)} product containers on this page.")

    for item in items:
        title_elem = item.find('p',class_='product-title')
        if not title_elem:
            title_elem = item.find('div', class_='title')
        if not title_elem:
            title_elem = item.find('a', class_='product-name')

        if title_elem:
            title = title_elem.get_text(strip=True)
        else:
            title = 'Unknown Title'

        price_elem = item.find('div',class_='product-price')

        if not price_elem:
            price_elem = item.find('span',class_='price')
        if not price_elem:
            price_elem = item.find('span',class_='actual-price')
        if price_elem:
            price_text = price_elem.get_text(strip=True)
            import re
            price_clean = re.sub(r'[^0-9.]','',price_text)

            try:
                price = float(price_clean)
            except ValueError:
                price = price_text
        else:
            price = 'N/A'

        products_list.append({
            'title':title,
            'price':price
        })
    return products_list
                                   

In [114]:
def save_to_csv(products, filename='kilimall_products.csv'):
    headers = ['Title','Price','Scrape_data']
    scrape_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    file_exists = Path(filename).exists()

    try:
        with open(filename,'a',newline='',encoding='utf-8')as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow(headers)
                logging.info(f"Created new file: {filename} with headers.")

            for product in products:
                row = [
                    product['title'],
                    product['price'],
                    scrape_time
                ]
                writer.writerow(row)

            logging.info(f"Appended {len(products)} products to {filename}")
    except Exceptions as e:
        logging.error(f"Failed to save CSV: {e}")
        
    

In [115]:
def send_price_alert(product_title, current_price, target_price, product_url):
    if not EMAIL_SENDER or not EMAIL_PASSWORD or not EMAIL_RECEIVER:
        logging.warning("Email credentials Missing.Skipping email alert.")
        return
    subject = f"PRICE DROP!! {product_title} is now below ${target_price}!"

    body = f"""
    Hello!

    Great news!!The item you are tracking has dropped in price.
    Product: {product_title}
    Current Price: KSh{current_price}
    Target Price: KSh{target_price}

    Click here to buy: {product_url}

    HAPPY SHOPPING!
    """

    msg = EmailMessage()
    msg['Subject'] = subject
    msg['From'] = EMAIL_SENDER
    msg['To'] = EMAIL_RECEIVER
    msg.set_content(body)

    try:
        with smtplib.SMTP_SSL('smtp.gmail.com',465) as server:
            server.login(EMAIL_SENDER,EMAIL_PASSWORD)
            server.send_message(msg)
            logging.info(f"Price alert email sent to {EMAIL_RECEIVER}")

    except Exception as e:
        logging.error(f"failed to send email:{e}")

In [116]:
def scrape_kilimall_category(category_url, max_pages=3,alert_threshold=1000):
    all_products = []

    for page in range(1, max_pages + 1):
        if '?' in category_url:
            page_url = f"{category_url}&page={page}"
        else:
            page_url = f"{category_url}?page={page}"
        logging.info(f"Scraping page {page}:{page_url}")
        soup = fetch_kilimall_page(page_url)

        if soup is None:
            logging.warning(f"Skipping page {page} due to fetch error.")
            continue
        page_products = extract_kilimall_products(soup)

        if not page_products:
            logging.info(f"No products found on page {page}.Stopping pagination")
            break
        all_products.extend(page_products)

        for product in page_products:
            if isinstance(product['price'],(int,float)):
                if product['price'] < alert_threshold:
                    logging.info(f"ALERT! {product['title']} is at KSh {product['price']} (below threshold)")
                    send_price_alert(
                        product_title=product['title'],
                        current_price=product['price'],
                        target_price=alert_threshold,
                        product_url=category_url
                    )
        sleep_time = random.uniform(2,5)
        logging.info(f"Waiting {sleep_time:2f} seconds before next page...")
        time.sleep(sleep_time)
        
    if all_products:
        save_to_csv(all_products)
        logging.info(f"Total products scraped: {len(all_products)}")
    else:
        logging.warning("No products were scraped at all.")
    return all_products

In [117]:
TEST_URL = 'https://www.kilimall.co.ke/search-result?id=2069&form=category&actName=TV,Audio&Video'
products_data = scrape_kilimall_category(
    category_url=TEST_URL,
    max_pages=2,
    alert_threshold=500
)
print("\n---SCRAPING COMPLETE ---")
print(f"Total items saved: {len(products_data)}")

2026-09-09 04:19:54,467 - INFO - Scraping page 1:https://www.kilimall.co.ke/search-result?id=2069&form=category&actName=TV,Audio&Video&page=1
2026-09-09 04:19:54,471 - INFO - Attempt 1 to fetch: https://www.kilimall.co.ke/search-result?id=2069&form=category&actName=TV,Audio&Video&page=1
2026-09-09 04:19:58,340 - INFO - Successfully fetched page: https://www.kilimall.co.ke/search-result?id=2069&form=category&actName=TV,Audio&Video&page=1
2026-09-09 04:19:58,348 - WARNING - No 'listing-item' found.Trying 'product-item'....
2026-09-09 04:19:58,356 - INFO - Found 5 product containers on this page.
2026-09-09 04:19:58,365 - INFO - ALERT! Thickened Style Hot Selling Unisex Messenger Bag Fashionabl Practical Small Bag Nylon Lightweight Small Crossbody Bag Versatile Single Shoulder Bag Orange is at KSh 199.0 (below threshold)
2026-09-09 04:19:59,657 - ERROR - failed to send email:(534, b'5.7.9 Application-specific password required. For more information, go to\n5.7.9  https://support.google.co


---SCRAPING COMPLETE ---
Total items saved: 10
